In [ ]:
# import necessary libraries needed for fine-tuning
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
from rouge_score import rouge_scorer
from bert_score import score
import pandas as pd
from datasets import Dataset
import evaluate

# load the preprocessed local CSVs
train_df = pd.read_csv("../data/cleaned/bbc/train.csv")
test_df = pd.read_csv("../data/cleaned/bbc/test.csv")

train_df_clean = train_df.dropna(subset=["Content", "Summary"]) 
test_df_clean = test_df.dropna(subset=["Content", "Summary"])

# convert pandas dataframe into a hugging face dataset object
train_dataset = Dataset.from_pandas(train_df_clean)
test_dataset = Dataset.from_pandas(test_df_clean)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("../models/bart-large-cnn")

# define tokenization function
def tokenize_function(dataset):
    # turns the input text into numerical tokens
    inputs = tokenizer(
        dataset["Content"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    # tokenize the labels (summary)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            dataset["Summary"],
            max_length=128,
            truncation=True,
            padding="max_length"
        )

    inputs["labels"] = labels["input_ids"]
    return inputs

# tokenizing the hugging face dataset object
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Load the pre-trained BART model
model = AutoModelForSeq2SeqLM.from_pretrained("../models/bart-large-cnn/")

# define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="../outputs/fine-tuned-bart/", # output directory for the train outputs
    evaluation_strategy="epoch", # tells the trainer when to run the evaluation strat
    save_strategy="epoch", # save the version after each epoch
    save_total_limit=2, # saves two versions
    load_best_model_at_end=True,
    metric_for_best_model="rougeL", 
    greater_is_better=True,
    learning_rate=3e-5, # controls how big of a step the optimizer takes when updating the model weights 
    per_device_train_batch_size=4, # number of samples processed in parallel per GPU during training
    per_device_eval_batch_size=8, # batch size for validation forward passes
    weight_decay=0.01, # Discourages weights from growing too large to prevent overfitting
    num_train_epochs=3, # number of full passes through the dataset
    predict_with_generate=True, # during evaluation, it makes the model generate summaries (instead of just computing loss)
    fp16=True, # Uses mixed precision training (16-bit floats) if GPU supports it (NVIDIA GPU with Tensor Cores). Speeds up training.
    logging_dir="../outputs/fine-tuned-bart/logs/",
    logging_strategy="epoch" 
)

# load the ROUGE score metric
rouge = evaluate.load("rouge")
bert = evaluate.load("bertscore")

# define compute_metrics() function
def compute_metrics(eval_pred):
    # generate predictions
    # 1. obtain the predictions and labels
    predictions, labels = eval_pred

    # decode to tokens into text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGE requires newline 
    decoded_preds = ["\n".join(pred.strip().split(". ")) for preds in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split(". ")) for label in decoded_labels]

    # compute ROUGE/BERTScore
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels,use_stemmer=True)
    bertscore_result = bertscore.compute(predictions=decoded_prefs, references=decoded_labels, lang="en")

    # return a single dictionary of metrics
    result = {
        "rouge1": rouge_result["rouge1"].mid.fmeasure,
        "rouge2": rouge_result["rouge2"].mid.fmeasure,
        "rougeL": rouge_result["rougeL"].mid.fmeasure,
        "bertscore_f1": np.mean(bertscore_result["f1"]),
    }

    # return in percent
    return {k: round(v * 100, 4) for k, v in result.items()}

# config the trainer
trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_test,
    tokenizer = tokenizer,
    compute_metrics = compute_metrics,
)

# run the trainer
trainer.train()

# evaluate the trainer
trainer.evaluate()

# save the fine-tuned model + tokenizer
model.save_pretrained("../outputs/fine-tuned-bart")
tokenizer.save_pretrained("../outputs/fine-tuned-bart")


Could not locate the tokenizer configuration file, will try to use the model config instead.
loading configuration file ../models/bart-large-cnn\config.json
Model config BartConfig {
  "_name_or_path": "../models/bart-large-cnn",
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_final_layer_norm": false,
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "early_stopping": true,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "force_bos_token_to_be_generated": true,
  "forced_bos_token_id": 0,
  "forced_eos_token_id": 2,
  "gradient_checkpointing": false,
  "id2label": {
 

KeyboardInterrupt: 